<a href="https://colab.research.google.com/github/jaqueuchoab/dados-divas/blob/main/join_censo_enade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Algoritmo de Junção das Bases**

Este algortimo faz a junção de duas bases específicas sendo elas ENADE e Censo da Educação Superior. A base ENADE passou por uma série de tratamentos até resultar em um arquivo final para esta junção, já a base da Educuação Superior permanece em sua forma original.

Assim como os demais algoritmos de tratamento este também foi desenvolvido com auxílio da *Inteligência Artificial Claude Sonnet 4.6* de forma supervisionada dado que os códigos foram revisados e validados por um ser humano.

**Importações de bibliotecas necessárias:**
- `pandas` — manipulação e compilação dos dados

In [ ]:
import pandas as pd

**Passo 1: Carregamento dos arquivos para junção**

Nesta seção, há o carregamento das bases Censo e ENADE no formato `.csv`. O arquivo ENADE passa por um tratamento nas colunas em que aparece uma sequência de caracteres geradas pelo BOM, mas que não interfere nos dados.

In [ ]:
# Carregamento do arquivo Censo
censo = pd.read_csv('/content/drive/MyDrive/microdados_unificacao_superior/enade_censo_2023/MICRODADOS_CADASTRO_CURSOS_2023.CSV', sep=';', encoding='latin-1')

# Carregamento do arquivo ENADE
enade_agrupado = pd.read_csv('/content/enade_agrupado_2023.csv', sep=';', encoding='utf-8-sig')
enade_agrupado.columns = enade_agrupado.columns.str.replace('ï»¿', '', regex=False).str.strip()

/tmp/ipykernel_1366/561943711.py:2: DtypeWarning: Columns (1,3,4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  censo = pd.read_csv('/content/drive/MyDrive/microdados_unificacao_superior/enade_censo_2023/MICRODADOS_CADASTRO_CURSOS_2023.CSV', sep=';', encoding='latin-1')


**Passo 2: Comparação entre os cursos e IES em cada uma das bases.**

In [ ]:
# Total de cursos no Censo
print(f"Total de linhas no Censo:     {len(censo)}")
print(f"Cursos no Censo:              {censo['CO_CURSO'].nunique()}")
print(f"IES no Censo:                 {censo['CO_IES'].nunique()}")

print(f"------------------------------------")

# Total de cursos no ENADE agrupado
print(f"Total de linhas no ENADE:     {len(enade_agrupado)}")
print(f"Cursos no ENADE:              {enade_agrupado['CO_CURSO'].nunique()}")
print(f"IES no ENADE:                 {enade_agrupado['CO_IES'].nunique()}")

Total de linhas no Censo:     671610
Cursos no Censo:              46317
IES no Censo:                 2580
------------------------------------
Total de linhas no ENADE:     9812
Cursos no ENADE:              9812
IES no ENADE:                 1347


**TESTE APAGAR DEPOIS**

In [ ]:
print(censo[['CO_CURSO', 'CO_MUNICIPIO', 'QT_MAT_FEM']].dtypes)
print(enade_agrupado[['CO_CURSO', 'CO_MUNIC_CURSO', 'QUANT_TP_GER_ALU_AUSENTE', 'QUANT_TP_DI_CE_ALU_AUSENTE']].dtypes)

CO_CURSO          int64
CO_MUNICIPIO    float64
QT_MAT_FEM        int64
dtype: object
CO_CURSO                      int64
CO_MUNIC_CURSO                int64
QUANT_TP_GER_ALU_AUSENTE      int64
QUANT_TP_DI_CE_ALU_AUSENTE    int64
dtype: object


**Passo 3: Merge das duas bases**

Nesta etapa há a junção dos arquivos Censo e ENADE. Visto que o Censo é a base da junção as variáveis agrupadas do ENADE são adicionados à esquerda

Depois do merge são removidas as colunas que foram duplicadas por terem o mesmo nome e estarem na mesma base, exemplo: `CO_IES`. Ou então porque tem os mesmos dados mas com nomes diferentes como é o caso de `CO_MUNICIPIO`

In [ ]:
base_join = censo.merge(
    enade_agrupado,
    left_on=['CO_CURSO', 'CO_MUNICIPIO'],
    right_on=['CO_CURSO', 'CO_MUNIC_CURSO'],
    how='left'
)

# Remoção de colunas duplicadas geradas pelo join
if 'CO_IES_x' in base_join.columns and 'CO_IES_y' in base_join.columns:
    base_join = base_join.drop(columns=['CO_IES_y'])
    base_join = base_join.rename(columns={'CO_IES_x': 'CO_IES'})

if 'CO_MUNIC_CURSO' in base_join.columns:
    base_join = base_join.drop(columns=['CO_MUNIC_CURSO'])

# Deve ter o mesmo tamanho do arquivo maior (pegar como referência o passo 2 que contém as quantidades de cada arquivo)
print(f"Total de linhas na base unificada:     {len(base_join)}")

Total de linhas na base unificada:     671610


**Passo 4: Tratamento tipo de dados variáveis ENADE**

Esta seção tem como objetivo *tentar* padronizar os tipos de dados em inteiro.

O contexto é que: Para cursos em que algumas variáveis ENADE não tem correspondências o dado é colocado como um `NaN`, e por mais que os dados das colunas ENADE venham como inteiros quando ocorre a junção elas são automaticamente convertidas para *float*.

A correção abaixo converte para `Int64`, com I maiúsculo, esse tipo aceita valores nulos o que permite que seja representada a *ausência de valor*, não apenas 0, pois poderia gerar uma interpretação errada, exemplo: Para o Curso X tiveram 0 alunos ausentes, sendo que o Curso X não estava presente na base ENADE.



In [ ]:
# Listando as colunas do arquivo ENADE agrupado
colunas_enade = enade_agrupado.columns.tolist()

# Para cada var do ENADE contida na Base Unificada será mudado o tipo para Int64, isso só é possível por termos a
# certeza de que todas colunas PODEM ser inteiros, uma vez que são contagens e médias.
for coluna in colunas_enade:
    if coluna in base_join.columns:
        try:
            base_join[coluna] = base_join[coluna].astype('Int64')
        except:
            pass

**TESTE APAGAR DEPOIS**

In [ ]:
print(base_join[['CO_CURSO', 'CO_MUNICIPIO', 'QT_MAT_FEM', 'QUANT_TP_GER_ALU_AUSENTE', 'QUANT_TP_DI_CE_ALU_AUSENTE']].dtypes)

CO_CURSO                        Int64
CO_MUNICIPIO                  float64
QT_MAT_FEM                      int64
QUANT_TP_GER_ALU_AUSENTE        Int64
QUANT_TP_DI_CE_ALU_AUSENTE      Int64
dtype: object


In [ ]:
base_join.to_csv('/content/join_censo_enade_2023.csv', sep=';',
                  encoding='utf-8-sig', index=False)

print(f"\nSUCESSO: Base unificada e salva!")


SUCESSO: Base unificada e salva!
